In [ ]:
from ase.io import read
from ase.visualize import view
import nqetools as nqe
# This follows:
# https://atomistic-cookbook.org/examples/pi-metad/pi-metad.html

import warnings

# Ignore all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# Make a directory to store everything
directory_opti = "opti"
directory_md = "md"
directory_meta_md = "meta_md"
directory_meta_pimd = "meta_pimd"

n_beads = 4
timestep = 1.0  # fs
total_steps = 5000
total_steps_md = 100

fix_com = True

stride = 10
temperature = 300
thermostat = 'smart_sampling_1ps_n6_w2'
md_type = "NVT-GLE"
driver_code = 'ase-mace'

# Expensive settings
driver_args = {'model': 'large',
               'device': 'cuda',
               'default_dtype': 'float64'}

# Cheap settings
driver_args = {'model': 'small',
               'device': 'cuda',
               'default_dtype': 'float32'}

# Plumed hills settings
n_bins = 100
stride_hills = 100
cv_limits = [-0.2, 0.2]
cv_limits = [None, None]

# # Plumed settings
plumed_type_opes = "opes-diff1"
plumed_args_opes = {'idx1': 1,
                    'idx2': 8,
                    'idx3': 0,
                    'barrier': 0.5,
                    'stride_hills': stride_hills, }

plumed_type_opes = "opes-dist"
plumed_args_opes = {'idx1': 0,
                    'idx2': 8,
                    'barrier': 0.5,
                    'stride_hills': stride_hills, }

plumed_type_opes = "opes-pt1"
plumed_args_opes = {'idx1': 0,
                    'idx2': 8,
                    'barrier': 0.5,
                    'stride_hills': stride_hills, }

In [ ]:
# Make the system
atoms = read("malonaldehyde.traj")
atoms.center(vacuum=10.0)
# Delete the hydrogen atom
del atoms[-1]
del atoms[5]

atoms = nqe.add_hydrogen_at_distance(atoms, 0, 1, 1.0)

view(atoms)

In [ ]:
# Run minimisation
output = nqe.run_optimise(directory_opti,
                          atoms,
                          driver=driver_code,
                          driver_args=driver_args)
atoms_opti, output_data_opti, output_desc_opti = output

In [ ]:
# Plot the energy of the minimisation
nqe.plot_step_energy(output_data_opti, save=False)

In [ ]:
# Run unbiased MD
output = nqe.run_md(directory_md,
                    atoms_opti,
                    driver=driver_code,
                    driver_args=driver_args,
                    total_steps=total_steps_md,
                    temperature=temperature,
                    timestep=timestep,
                    thermostat=thermostat,
                    md_type=md_type,
                    fix_com=fix_com,
                    stride=1,
                    n_beads=1)
atoms_md, output_data_md, output_desc_md = output

In [ ]:
# Run OPES metadynamics
output = nqe.run_plumed_md(directory_meta_md,
                           atoms_md,
                           driver=driver_code,
                           driver_args=driver_args,
                           total_steps=total_steps,
                           temperature=temperature,
                           timestep=timestep,
                           thermostat=thermostat,
                           md_type=md_type,
                           fix_com=fix_com,
                           stride=stride,
                           n_beads=1,
                           plumed_type=plumed_type_opes,
                           plumed_args=plumed_args_opes)
atoms_meta_md, output_data_meta_md, output_desc_meta_md = output

In [ ]:
view(atoms_meta_md)

In [ ]:
# Plot the time evolution of the energy and bias
nqe.plot_time_potential_bias(output_data_meta_md, save=False)

In [ ]:
# Plot the energy conservation
nqe.plot_time_energy_conservation(output_data_meta_md, save=False)

In [ ]:
# Plot the time evolution of the temperature
nqe.plot_time_temperature(output_data_meta_md, save=False)

In [ ]:
# Run the hills command
nqe.run_plumed_hills_opes(directory_meta_md,
                          temperature=temperature,
                          bins=n_bins,
                          cv=cv_limits)
# Plot the free energy surface convergence
fes_arrays_meta_md = nqe.load_fes_data(directory_meta_md, n_bins)
fes_times = nqe.get_fes_times(timestep, total_steps, fes_arrays_meta_md)
nqe.plot_fes_series_1d(fes_arrays_meta_md, fes_times)

In [ ]:
# Run PIMD OPES metadynamics
output = nqe.run_plumed_md(directory_meta_pimd,
                           atoms_md,
                           driver=driver_code,
                           driver_args=driver_args,
                           total_steps=total_steps,
                           temperature=temperature,
                           timestep=timestep,
                           thermostat=thermostat,
                           md_type=md_type,
                           fix_com=fix_com,
                           stride=stride,
                           n_beads=n_beads,
                           plumed_type=plumed_type_opes,
                           plumed_args=plumed_args_opes)
atoms_meta_pimd, output_data_meta_pimd, output_desc_meta_pimd = output

In [ ]:
# Plot the time evolution of the energy and bias
nqe.plot_time_potential_bias(output_data_meta_pimd, save=False)

In [ ]:
# Plot the energy conservation
nqe.plot_time_energy_conservation(output_data_meta_pimd, save=False)

In [ ]:
# Plot the time evolution of the temperature
nqe.plot_time_temperature(output_data_meta_pimd, save=False)

In [ ]:
# Run the hills command
nqe.run_plumed_hills_opes(directory_meta_pimd,
                          temperature=temperature,
                          bins=n_bins,
                          cv=cv_limits)

# Plot the free energy surface convergence
fes_arrays_meta_pimd = nqe.load_fes_data(directory_meta_pimd, n_bins)
fes_times = nqe.get_fes_times(timestep, total_steps, fes_arrays_meta_pimd)
nqe.plot_fes_series_1d(fes_arrays_meta_pimd, fes_times)

In [ ]:
# Plot the comparison between MD and PIMD, compare the last frame
nqe.plot_fes_series_1d_compare(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])

In [ ]:
# Plot the comparison between MD and PIMD, compare the last frame
nqe.plot_fes_contourf_compare(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])

In [ ]:
nqe.plot_fes_sep(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])

In [ ]:
# Plot the comparison between MD and PIMD, compare the last frame
nqe.plot_fes_contourf_compare(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])

In [ ]:
nqe.plot_fes_sep(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])